In [ ]:
try:
    import yfinance as yf
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance"])
    import yfinance as yf

from datetime import datetime, timedelta


def ingest_data(ticker, interval):
    end = datetime.now()
    start = end - timedelta(hours=1)
    df = yf.download(ticker, start=start, end=end, interval=interval, auto_adjust=False)

    first_price = df["Close"].iloc[0]
    last_price = df["Close"].iloc[-1]

    return spark.createDataFrame(
        [(
            ticker,
            df.index.min(),
            df.index.max(),
            float(df["Close"].max()),
            float(df["Close"].min()),
            float(last_price),
            float((last_price - first_price) / first_price * 100),
            float((df["Close"] * df["Volume"]).sum() / df["Volume"].sum()),
            float(df["Volume"].sum()),
        )],
        [
            "ticker", "window_start", "window_end", "high",
            "low", "last_price", "pct_change", "vwap", "volume"
        ]
    )

In [ ]:
def update_config(data, params):
    return params